In [ ]:
# 1. Install required packages (100% Offline Mode)
import os
import sys
import subprocess
import glob

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

offline_pkg_dir = "/kaggle/input/datasets/mduy2911/offline-packages"
print(f"Installing offline packages from: {offline_pkg_dir}...")
wheels = glob.glob(os.path.join(offline_pkg_dir, "*.whl"))

if wheels:
    cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps"] + wheels
    subprocess.run(cmd, check=True)
    print("Offline installation completed successfully!")
else:
    raise FileNotFoundError(f"No offline wheels found in {offline_pkg_dir}!")

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"


In [ ]:
# 2. TRAINING HYPERPARAMETERS (Optimized for RTX Pro 6000 Ada / 96GB VRAM)
import os

MODEL_ID = "/kaggle/input/datasets/mduy2911/model-cache"

# --- Hardware Optimizations ---
USE_QLORA = False
GRADIENT_CHECKPOINTING = True

# --- Training Hyperparameters (2-Phase Router + Physics Strategy) ---
MAX_LENGTH = 4096            # Optimized sequence length covering >99.9% of Physics & 100% of Router
BATCH_SIZE = 8               # Physical batch size optimized for RTX 6000
GRADIENT_ACCUMULATION = 4   # Gradient accumulation steps (Effective batch size = 32)

EPOCHS_P1 = 4               # Phase 1: Physics Focus epochs
LEARNING_RATE_P1 = 1e-4     # Phase 1: Physics Focus learning rate
OUTPUT_DIR_P1 = "/kaggle/working/results_router_physics_p1"

EPOCHS_P2 = 4               # Phase 2: Router Focus epochs
LEARNING_RATE_P2 = 5e-5     # Phase 2: Router Focus learning rate (lower to preserve Physics weights)
OUTPUT_DIR_P2 = "/kaggle/working/results_router_physics_p2"

os.environ["WANDB_MODE"] = "disabled"
USE_WANDB = False


In [ ]:
# 3. Load Physics and Router Datasets
import os
import json

physics_path = "/kaggle/input/datasets/mduy2911/folc-train/physics_distillation.json"
router_path = "/kaggle/input/datasets/mduy2911/folc-train/router_dataset.json"

def load_physics_dataset(path):
    samples = []
    if not path or not os.path.exists(path):
        print(f"Warning: Path '{path}' does not exist. Skipping.")
        return []
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    for item in data:
        inp = item.get("input", "")
        out = item.get("output", "")
        q = item.get("question", "")
        if inp and out:
            samples.append({"input": inp, "output": out, "question": q.strip()})
    print(f"Loaded {len(samples)} physics samples from {os.path.basename(path)}")
    return samples

def load_router_dataset(path):
    samples = []
    if not path or not os.path.exists(path):
        print(f"Warning: Path '{path}' does not exist. Skipping.")
        return []
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    for item in data:
        inp = item.get("input", "")
        out = item.get("output", "")
        if inp and out:
            samples.append({"input": inp, "output": out, "question": inp.strip()})
    print(f"Loaded {len(samples)} router samples from {os.path.basename(path)}")
    return samples

physics_samples = load_physics_dataset(physics_path)
router_samples = load_router_dataset(router_path)


In [ ]:
# 4. Define Prompt Templates
physics_system_prompt = """You are a precise physics reasoning agent.

TASK:
Convert a physics problem and any provided reasoning policies into executable SymPy code and a valid JSON response.

<REASONING_POLICY_OVERRIDE>

A <reasoning_policies> block may be provided.

If present:

1. Treat it as the primary reasoning guidance.

2. Follow its definitions, representations, topology rules,
   coordinate conventions, state models, and solution procedures.

3. Apply the underlying reasoning pattern rather than copying
   example expressions verbatim.

4. Override a policy only when required for:
   - physical validity
   - mathematical validity
   - SymPy executability

</REASONING_POLICY_OVERRIDE>

<OPERATING_CONSTRAINTS>

Return ONLY:

{
  "thought": "...",
  "physics_analysis": [...],
  "algebraic_reasoning": [...],
  "python_code": "...",
  "json_terminated": true
}

thought format:

<detected structure>. <activated reasoning pattern>. <solution strategy>.

thought:
- concise
- high-level
- no calculations
- no intermediate values
- no final values

physics_analysis:
- concise policy-grounded physical interpretation
- record relevant physical facts, assumptions, states, or constraints
- no calculations
- no final values

algebraic_reasoning:
- concise policy-grounded symbolic procedure
- describe the intended solve workflow
- no calculations
- no final values

python_code:
- begin with "import sympy as sp"
- use sp.Float(...) for numerical constants
- define variables before use
- solve symbolically before numeric evaluation
- evaluate norms, distances, and square roots using float(...) or .evalf()
- single-line string only
- separate statements using "; "
- no loops
- no functions
- no conditional branches
- no newline characters

Final code statements must define:

ans = [...]
unit = [...]

Requirements:
- ans must be a list
- unit must be a list
- len(ans) == len(unit)
- use raw SymPy-computed values
- do not manually round or format values
- no trailing semicolon after the final statement

Use SI base or SI derived units only.
Do not use engineering-prefix units.

Return the JSON object only.

</OPERATING_CONSTRAINTS>""".strip()

router_system_prompt = """# Role
You are an expert Physics and Mathematical Reasoner acting as a strict Semantic Classification Router. Your sole task is to analyze problems and map them to their exact domains and operational states based on the rules below, without solving the problem itself.

# Router Configuration

## Domain Options
Physics domains:
- electrostatic_force
- electrostatic_field
- ac_impedance
- resonance
- frequency_scaling
- electromagnetism
- oscillation_energy
- circuit_power
- capacitance_and_energy
- experimental_physics

Reasoning domains:
- spatial_vector_geometry
- qualitative_reasoning
- symbolic_derivation
- proportional_scaling
- error_analysis

## Domain Selection Rules

### spatial_vector_geometry
MANDATORY when: The problem text introduces spatial positioning, coordinate paths, or multi-point labels (such as points A, B, and C) that dictate relative distances or non-collinear structures. 
CRITICAL: If a core physics domain applies to a system with multiple non-co-located spatial coordinates, you must output this domain alongside the physics domain. Do not omit it.
- IMPLICATION: Directs the downstream logic to parse spatial and matrix vectors.

### experimental_physics
Use when: Experimental measurements, trial data sets, absolute or relative uncertainties, instrument tolerances, or duplicate measurement values.

### capacitance_and_energy
Use when: Parallel-plate configurations, dielectric replacement/materials, electrostatic potential energy storage. Look for variables normalized strictly to base Coulombs (e.g., 'e-6 C', 'e-7 C'), base Farads (e.g., 'e-5 F', 'F'), or area/separations scaled to meters or square meters (e.g., 'm', 'm^2').

### error_analysis
Use when: Random error, percentage relative error, standard deviation, range-based uncertainty, or error propagation.

### proportional_scaling
Use when: The problem relies on qualitative ratio behaviors, relative trends, or fractional multiplier adjustments (e.g., "if the resistance doubles", "halves", or "is cut in a 1:3 ratio") without presenting absolute initial values.
- IMPLICATION: Directs downstream logic to evaluate mathematical behavior scaling trends instead of raw value evaluations.

### electrostatic_force
Use when: Coulomb interactions, electric force, attractive/repulsive forces, or point charge mechanics.

### electrostatic_field
Use when: Electric field intensity (E), flux lines, field distribution, or co-located vertex field calculations.

### ac_impedance
Use when: RLC alternating current series circuits, phasor geometry, component voltages, or phase angle calculations. Look for variables normalized to base Ohm units ('ohm') and alternating current frequencies normalized strictly to Hertz ('Hz').

### resonance
Use when: Reactance cancellations (XL == XC), maximum AC current states, zero phase angles, or resonant frequency calculations.

### frequency_scaling
Use when: System frequency shifts, omega transformations, or non-linear adjustments to reactance. Look for multiple sequential frequencies scaled strictly to base Hertz ('Hz').

### electromagnetism
Use when: Magnetic flux, flux linkage, inductive EMF, self-inductance (L), solenoids, or magnetic energy density.

### oscillation_energy
Use when: LC circuit energy conservation, continuous exchange between electric and magnetic energies, or fraction of maximum energy states.

### circuit_power
Use when: DC power, AC average power, power factors, Joule heating, or parallel multi-branch entity calculations.

### qualitative_reasoning
Use when: The question is entirely conceptual, descriptive, or requires a binary confirmation, but completely lacks mathematical metrics, percentages, multipliers, or numerical scale factors.
- IMPLICATION: Directs downstream logic to output a raw text description list or a strict "Yes"/"No" string response.

## Additional Field
### multi_state
Set true when problem contains: before/after states, transformed systems, frequency changes, or state transitions. Otherwise set false.

## OUTPUT FORMAT
{
  "domains": ["domain1", "domain2"],
  "multi_state": true
}

## Strict Operational Directives:
- You must output EXACTLY one valid JSON object.
- Absolutely NO markdown wrapping (do not use ```json or ```).
- Absolutely NO explanations, introductory text, or closing text.
- Do not include chain-of-thought processing or reasoning text in the output.
  
## Structural Pairing Rules
- For any problem describing explicit coordinates... the "domains" array 
  MUST contain the primary physics domain and "spatial_vector_geometry". 
  A third domain may be added when applicable (e.g., "symbolic_derivation").
- CONCEPTUAL PAIRING GUARD: If a problem completely lacks relative physical distances, layout structures, or spatial coordinates (e.g., purely descriptive circuit loops or trend predictions), you are strictly BANNED from outputting "spatial_vector_geometry". Instead, you must pair the physics domain with its corresponding conceptual reasoning framework:
  - If it features text trend movements or binary attributes: ["physics_domain", "qualitative_reasoning"]
  - If it features multiplier/fractional factors ("doubles", "halves"): ["physics_domain", "proportional_scaling"]
- BANNED: Never drop a required reasoning domain for multi-variable or structural configurations. Never output a single-element array for spatial layout problems. Never output more than three domain elements under any circumstances.
- BANNED: Do not use the instruction "Prefer specific domains over broad ones."""


In [ ]:
# 5. Format Dataset (Chat Template) and split Train/Val
import os
import random
from datasets import Dataset
from transformers import AutoTokenizer

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, 
    trust_remote_code=True, 
    local_files_only=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Grouped split by question text to prevent cross-task data leakage
all_questions = set()
for item in physics_samples:
    all_questions.add(item["question"])
for item in router_samples:
    all_questions.add(item["question"])

all_questions = sorted(list(all_questions))
random.Random(42).shuffle(all_questions)

split_idx = int(len(all_questions) * 0.9)
train_questions = set(all_questions[:split_idx])
val_questions = set(all_questions[split_idx:])

train_physics = [item for item in physics_samples if item["question"] in train_questions]
val_physics = [item for item in physics_samples if item["question"] in val_questions]

train_router = [item for item in router_samples if item["question"] in train_questions]
val_router = [item for item in router_samples if item["question"] in val_questions]

def format_samples(physics_list, router_list):
    formatted = []
    
    # Format Physics samples
    for item in physics_list:
        inp = item["input"]
        out = item["output"]
        formatted.append({
            "messages": [
                {"role": "system", "content": physics_system_prompt},
                {"role": "user", "content": inp.strip()},
                {"role": "assistant", "content": out.strip()}
            ]
        })
        
    # Format Router samples
    for item in router_list:
        inp = item["input"]
        out = item["output"]
        formatted.append({
            "messages": [
                {"role": "system", "content": router_system_prompt},
                {"role": "user", "content": f"<QUESTION>\n{inp.strip()}\n</QUESTION>"},
                {"role": "assistant", "content": out.strip()}
            ]
        })
        
    return formatted

def apply_template(batch):
    texts = []
    for messages in batch["messages"]:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

# Common Validation dataset
formatted_val = format_samples(val_physics, val_router)
val_dataset = Dataset.from_list(formatted_val)
val_dataset = val_dataset.map(apply_template, batched=True, remove_columns=["messages"])
val_dataset = val_dataset.shuffle(seed=42)

# Phase 1 Train dataset: 100% Physics + 20% Router
num_router_p1 = int(len(train_router) * 0.20)
train_router_p1 = train_router[:num_router_p1]
formatted_train_p1 = format_samples(train_physics, train_router_p1)
train_dataset_p1 = Dataset.from_list(formatted_train_p1)
train_dataset_p1 = train_dataset_p1.map(apply_template, batched=True, remove_columns=["messages"])
train_dataset_p1 = train_dataset_p1.shuffle(seed=42)

# Phase 2 Train dataset: 100% Router + 50% Physics
num_physics_p2 = int(len(train_physics) * 0.50)
train_physics_p2 = random.Random(42).sample(train_physics, num_physics_p2)
formatted_train_p2 = format_samples(train_physics_p2, train_router)
train_dataset_p2 = Dataset.from_list(formatted_train_p2)
train_dataset_p2 = train_dataset_p2.map(apply_template, batched=True, remove_columns=["messages"])
train_dataset_p2 = train_dataset_p2.shuffle(seed=42)

print(f"Physics train={len(train_physics)}, val={len(val_physics)}")
print(f"Router train={len(train_router)}, val={len(val_router)}")
print(f"Phase 1 Train size: {len(train_dataset_p1)}")
print(f"Phase 2 Train size: {len(train_dataset_p2)}")
print(f"Common Validation size: {len(val_dataset)}")


In [ ]:
# 6. Custom Collator, JSON Cleaner, and Evaluation Metrics
import re
import random
import torch
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback, DataCollatorForLanguageModeling, EarlyStoppingCallback
from peft import LoraConfig, get_peft_model, PeftModel

class LossLoggingCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            if "loss" in logs:
                print(f"Step {state.global_step}: training_loss = {logs['loss']:.6f}")
            if "eval_loss" in logs:
                print(f"Step {state.global_step}: validation_loss = {logs['eval_loss']:.6f}")

class CustomDataCollator(DataCollatorForLanguageModeling):
    def __init__(self, response_template, tokenizer, *args, **kwargs):
        super().__init__(tokenizer=tokenizer, mlm=False, *args, **kwargs)
        self.response_template = response_template
        self.response_token_ids = self.tokenizer.encode(self.response_template, add_special_tokens=False)
        
    def __call__(self, examples):
        batch = super().__call__(examples)
        labels = batch["labels"]
        for i in range(labels.size(0)):
            input_ids = batch["input_ids"][i].tolist()
            response_idx = -1
            n_template = len(self.response_token_ids)
            for j in range(len(input_ids) - n_template + 1):
                if input_ids[j:j+n_template] == self.response_token_ids:
                    response_idx = j + n_template
                    break
            
            if response_idx != -1:
                labels[i, :response_idx] = -100
                
        return batch

def clean_json_response(response: str) -> str:
    response = response.strip()
    if response.startswith("```"):
        match = re.search(r"```(?:json)?\s*(.*?)\s*```", response, re.DOTALL)
        if match:
            response = match.group(1).strip()
            
    first_brace = response.find("{")
    first_bracket = response.find("[")
    
    if first_brace != -1 and (first_bracket == -1 or first_brace < first_bracket):
        obj_match = re.search(r"(\{.*\})", response, re.DOTALL)
        if obj_match:
            response = obj_match.group(1).strip()
        else:
            obj_match_open = re.search(r"(\{.*)", response, re.DOTALL)
            if obj_match_open:
                response = obj_match_open.group(1).strip()
    elif first_bracket != -1:
        array_match = re.search(r"(\[.*\])", response, re.DOTALL)
        if array_match:
            response = array_match.group(1).strip()
        else:
            array_match_open = re.search(r"(\[.*)", response, re.DOTALL)
            if array_match_open:
                response = array_match_open.group(1).strip()
                
    in_quote = False
    escape = False
    stack = []
    i = 0
    while i < len(response):
        char = response[i]
        if escape:
            escape = False
        elif char == '\\':
            escape = True
        elif char == '"':
            in_quote = not in_quote
        elif not in_quote:
            if char in ('{', '['):
                stack.append(char)
            elif char in ('}', ']'):
                if stack and ((char == '}' and stack[-1] == '{') or (char == ']' and stack[-1] == '[')):
                    stack.pop()
        i += 1
        
    if in_quote:
        response += '"'
    while stack:
        top = stack.pop()
        if top == '{':
            response += '}'
        elif top == '[':
            response += ']'
            
    return response

def compare_physics_answers(pred_ans, pred_unit, gt_ans, gt_unit):
    if pred_ans is None or gt_ans is None:
        return False
    if not isinstance(pred_ans, list):
        pred_ans = [pred_ans]
    if not isinstance(gt_ans, list):
        gt_ans = [gt_ans]
    if len(pred_ans) != len(gt_ans):
        return False
    for p_val, g_val in zip(pred_ans, gt_ans):
        try:
            p_num = float(p_val)
            g_num = float(g_val)
            if g_num == 0.0:
                if abs(p_num) >= 1e-5:
                    return False
            else:
                rel_err = abs(p_num - g_num) / abs(g_num)
                if rel_err > 0.02:
                    return False
        except (ValueError, TypeError):
            if str(p_val).strip().lower() != str(g_val).strip().lower():
                return False
    return True

def evaluate_physics_accuracy(model, tokenizer, val_samples, limit=None):
    eval_limit = limit if limit is not None else len(val_samples)
    print(f"Evaluating Physics Accuracy on {eval_limit} samples...")
    correct_count = 0
    total_count = 0
    valid_json_count = 0
    python_syntax_count = 0
    exec_count = 0
    
    failed_cases = []
    
    if limit is not None:
        eval_subset = random.Random(42).sample(val_samples, min(len(val_samples), limit))
    else:
        eval_subset = val_samples
        
    for item in eval_subset:
        inp = item["input"]
        gt_output_str = item["output"]
        
        messages = [
            {"role": "system", "content": physics_system_prompt},
            {"role": "user", "content": inp.strip()}
        ]
        
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=1024,
                do_sample=False,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
            
        generated_ids = outputs[0][inputs.input_ids.shape[1]:]
        response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        cleaned_response = clean_json_response(response)
        
        is_json_valid = False
        is_syntax_valid = False
        is_executed = False
        is_correct = False
        pred_ans = None
        pred_unit = None
        gt_ans = None
        gt_unit = None
        
        try:
            parsed_response = json.loads(cleaned_response)
            valid_json_count += 1
            is_json_valid = True
            code_str = parsed_response.get("python_code", "")
            
            try:
                gt_parsed = json.loads(gt_output_str)
                gt_code_str = gt_parsed.get("python_code", "")
            except Exception:
                gt_code_str = ""
                
            if code_str:
                try:
                    compile(code_str, "<string>", "exec")
                    python_syntax_count += 1
                    is_syntax_valid = True
                except Exception:
                    pass
                    
                if gt_code_str:
                    local_vars_pred = {}
                    local_vars_gt = {}
                    try:
                        import sympy as sp
                        exec(code_str, {"sp": sp}, local_vars_pred)
                        pred_ans = local_vars_pred.get("ans", None)
                        pred_unit = local_vars_pred.get("unit", None)
                        
                        exec(gt_code_str, {"sp": sp}, local_vars_gt)
                        gt_ans = local_vars_gt.get("ans", None)
                        gt_unit = local_vars_gt.get("unit", None)
                        
                        if pred_ans is not None and gt_ans is not None:
                            exec_count += 1
                            is_executed = True
                            if compare_physics_answers(pred_ans, pred_unit, gt_ans, gt_unit):
                                correct_count += 1
                                is_correct = True
                    except Exception:
                        pass
        except Exception:
            pass
            
        if not is_correct:
            failed_cases.append({
                "input": inp,
                "output_gt": gt_output_str,
                "raw_response": response,
                "cleaned_response": cleaned_response,
                "pred_ans": pred_ans,
                "pred_unit": pred_unit,
                "gt_ans": gt_ans,
                "gt_unit": gt_unit,
                "is_json_valid": is_json_valid,
                "is_syntax_valid": is_syntax_valid,
                "is_executed": is_executed
            })
        total_count += 1
        
    acc = (correct_count / total_count) * 100 if total_count > 0 else 0
    json_rate = (valid_json_count / total_count) * 100 if total_count > 0 else 0
    python_syntax_rate = (python_syntax_count / total_count) * 100 if total_count > 0 else 0
    exec_rate = (exec_count / total_count) * 100 if total_count > 0 else 0
    
    print("\n=== Physics Evaluation Metrics ===")
    print(f"Physics Accuracy: {acc:.2f}% ({correct_count}/{total_count})")
    print(f"Physics JSON Validity Rate: {json_rate:.2f}% ({valid_json_count}/{total_count})")
    print(f"Physics Python Syntax Validity Rate: {python_syntax_rate:.2f}% ({python_syntax_count}/{total_count})")
    print(f"Physics Code Execution Rate: {exec_rate:.2f}% ({exec_count}/{total_count})")
    
    # Save failure cases to a file
    output_failed_path = "physics_failed_cases.json"
    class SympyEncoder(json.JSONEncoder):
        def default(self, obj):
            try:
                import sympy as sp
                if isinstance(obj, sp.Basic):
                    try:
                        val = float(obj.evalf())
                        return int(val) if val.is_integer() else val
                    except Exception:
                        return str(obj)
            except Exception:
                pass
            if isinstance(obj, set):
                return list(obj)
            try:
                return super().default(obj)
            except TypeError:
                return str(obj)

    try:
        with open(output_failed_path, "w", encoding="utf-8") as f_out:
            json.dump(failed_cases, f_out, indent=2, ensure_ascii=False, cls=SympyEncoder)
        print(f"Saved {len(failed_cases)} failed cases to: {os.path.abspath(output_failed_path)}")
    except Exception as save_err:
        print(f"Error saving failed cases: {save_err}")
        
    return acc, json_rate, python_syntax_rate, exec_rate

def evaluate_router_accuracy(model, tokenizer, val_samples, limit=None):
    eval_limit = limit if limit is not None else len(val_samples)
    print(f"Evaluating Router Accuracy on {eval_limit} samples...")
    correct_count = 0
    total_count = 0
    valid_json_count = 0
    domain_exact_match_count = 0
    multi_state_correct_count = 0
    
    failed_cases = []
    
    if limit is not None:
        eval_subset = random.Random(42).sample(val_samples, min(len(val_samples), limit))
    else:
        eval_subset = val_samples
        
    for item in eval_subset:
        inp = item["input"]
        gt_output_str = item["output"]
        
        messages = [
            {"role": "system", "content": router_system_prompt},
            {"role": "user", "content": f"<QUESTION>\n{inp.strip()}\n</QUESTION>"}
        ]
        
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
            
        generated_ids = outputs[0][inputs.input_ids.shape[1]:]
        response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        cleaned_response = clean_json_response(response)
        
        is_json_valid = False
        is_correct = False
        pred_domains = []
        gt_domains = []
        pred_multi_state = False
        gt_multi_state = False
        
        try:
            parsed_response = json.loads(cleaned_response)
            valid_json_count += 1
            is_json_valid = True
            
            # Ground truth
            gt_parsed = json.loads(gt_output_str)
            
            pred_domains = parsed_response.get("domains", [])
            gt_domains = gt_parsed.get("domains", [])
            
            pred_multi_state = parsed_response.get("multi_state", False)
            gt_multi_state = gt_parsed.get("multi_state", False)
            
            domain_exact_match = sorted(pred_domains) == sorted(gt_domains)
            if domain_exact_match:
                domain_exact_match_count += 1
                
            multi_state_correct = bool(pred_multi_state) == bool(gt_multi_state)
            if multi_state_correct:
                multi_state_correct_count += 1
                
            if domain_exact_match and multi_state_correct:
                correct_count += 1
                is_correct = True
        except Exception:
            pass
            
        if not is_correct:
            failed_cases.append({
                "input": inp,
                "output_gt": gt_output_str,
                "raw_response": response,
                "cleaned_response": cleaned_response,
                "pred_domains": pred_domains,
                "gt_domains": gt_domains,
                "pred_multi_state": pred_multi_state,
                "gt_multi_state": gt_multi_state,
                "is_json_valid": is_json_valid
            })
            
        total_count += 1
        
    acc = (correct_count / total_count) * 100 if total_count > 0 else 0
    json_rate = (valid_json_count / total_count) * 100 if total_count > 0 else 0
    domain_acc = (domain_exact_match_count / total_count) * 100 if total_count > 0 else 0
    multi_state_acc = (multi_state_correct_count / total_count) * 100 if total_count > 0 else 0
    
    print("\n=== Router Evaluation Metrics ===")
    print(f"Router Exact Match Accuracy: {acc:.2f}% ({correct_count}/{total_count})")
    print(f"Router JSON Validity Rate: {json_rate:.2f}% ({valid_json_count}/{total_count})")
    print(f"Router Domain Exact Match Rate: {domain_acc:.2f}% ({domain_exact_match_count}/{total_count})")
    print(f"Router Multi-State Accuracy: {multi_state_acc:.2f}% ({multi_state_correct_count}/{total_count})")
    
    # Save failure cases to a file
    output_failed_path = "router_failed_cases.json"
    class SympyEncoder(json.JSONEncoder):
        def default(self, obj):
            try:
                import sympy as sp
                if isinstance(obj, sp.Basic):
                    try:
                        val = float(obj.evalf())
                        return int(val) if val.is_integer() else val
                    except Exception:
                        return str(obj)
            except Exception:
                pass
            if isinstance(obj, set):
                return list(obj)
            try:
                return super().default(obj)
            except TypeError:
                return str(obj)

    try:
        with open(output_failed_path, "w", encoding="utf-8") as f_out:
            json.dump(failed_cases, f_out, indent=2, ensure_ascii=False, cls=SympyEncoder)
        print(f"Saved {len(failed_cases)} failed cases to: {os.path.abspath(output_failed_path)}")
    except Exception as save_err:
        print(f"Error saving failed cases: {save_err}")
        
    return acc, json_rate, domain_acc, multi_state_acc

In [ ]:
# 7. Train Model Helper Function
import sys
import gc

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Check hardware bfloat16 support
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    compute_dtype = torch.bfloat16
    use_bf16 = True
    use_fp16 = False
    print("GPU supports bfloat16. Using bfloat16 compute.")
else:
    compute_dtype = torch.float16
    use_bf16 = False
    use_fp16 = True
    print("Using float16 compute.")

def clean_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("GPU and system memory cleaned.")

target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
peft_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

def load_base_model():
    print("Loading a fresh instance of the base model...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=compute_dtype if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
        local_files_only=True
    )
    model.config.use_cache = False
    return model

from transformers import AutoModelForCausalLM

def train_model(train_dataset, val_dataset, output_dir, epochs=2, learning_rate=1e-4, resume_from_dir=None):
    clean_memory()
    
    num_samples = len(train_dataset)
    effective_batch_size = BATCH_SIZE * GRADIENT_ACCUMULATION
    steps_per_epoch = num_samples // effective_batch_size
    if num_samples % effective_batch_size != 0:
        steps_per_epoch += 1
    total_steps = steps_per_epoch * epochs
    warmup_steps = max(1, int(total_steps * 0.05))
    print(f"Training on {num_samples} samples. Steps per epoch: {steps_per_epoch}. Total steps: {total_steps}. Dynamic warmup steps: {warmup_steps}")
    
    base_model = load_base_model()
    
    load_dir = resume_from_dir if resume_from_dir else output_dir
    adapter_config_path = os.path.join(load_dir, "adapter_config.json")
    if os.path.exists(adapter_config_path):
        print(f"Loading PEFT adapter weights from {load_dir}...")
        model = PeftModel.from_pretrained(base_model, load_dir, is_trainable=True)
    else:
        print("Initializing a new PEFT adapter...")
        model = get_peft_model(base_model, peft_config)
        
    model.print_trainable_parameters()
    
    sft_config = SFTConfig(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=learning_rate,
        lr_scheduler_type="cosine",
        warmup_steps=warmup_steps,
        fp16=use_fp16,
        bf16=use_bf16,
        logging_steps=1,
        logging_first_step=True,
        optim="adamw_torch_fused",
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        per_device_eval_batch_size=BATCH_SIZE,
        report_to="none",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=2,
        dataset_text_field="text",
        max_length=MAX_LENGTH,
        dataloader_num_workers=2,
        dataloader_pin_memory=True
    )

    response_template = "<|im_start|>assistant\n"
    collator = CustomDataCollator(
        response_template=response_template, 
        tokenizer=tokenizer
    )
    
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        args=sft_config,
        data_collator=collator,
        callbacks=[LossLoggingCallback(), EarlyStoppingCallback(early_stopping_patience=2)]
    )

    checkpoints = glob.glob(os.path.join(output_dir, "checkpoint-*"))
    resume_path = None
    if checkpoints:
        checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
        resume_path = checkpoints[-1]
        print(f"Found checkpoints. Resuming training from: {resume_path}")
        
    trainer.train(resume_from_checkpoint=resume_path)
    
    print(f"Saving best validation adapter weights and tokenizer to {output_dir}...")
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print("Training completed successfully!")
    
    try:
        print("\n=== Starting Post-Training Accuracy Evaluation ===")
        model.eval()
        evaluate_physics_accuracy(model, tokenizer, val_physics, limit=None)
        evaluate_router_accuracy(model, tokenizer, val_router, limit=None)
    except Exception as e:
        print(f"Error during post-training evaluation: {e}")
        
    del trainer
    del model
    del base_model
    clean_memory()


In [ ]:
# 8. Start Training (2-Phase Strategy)
print("=== STARTING PHASE 1 (Physics focus: 100% Physics : 20% Router) ===")
train_model(
    train_dataset=train_dataset_p1,
    val_dataset=val_dataset,
    output_dir=OUTPUT_DIR_P1,
    epochs=EPOCHS_P1,
    learning_rate=LEARNING_RATE_P1,
    resume_from_dir=None
)

print("\n=== STARTING PHASE 2 (Router focus: 100% Router : 50% Physics) ===")
train_model(
    train_dataset=train_dataset_p2,
    val_dataset=val_dataset,
    output_dir=OUTPUT_DIR_P2,
    epochs=EPOCHS_P2,
    learning_rate=LEARNING_RATE_P2,
    resume_from_dir=OUTPUT_DIR_P1
)


In [ ]:
# 9. Zip training results
import shutil
import os

OUTPUT_DIR = OUTPUT_DIR_P2
zip_name = "/kaggle/working/results_router_physics"

if os.path.exists(OUTPUT_DIR):
    print("Zipping results folder...")
    shutil.make_archive(zip_name, 'zip', OUTPUT_DIR)
    print(f"Successfully zipped to: {zip_name}.zip")
else:
    print(f"Results directory {OUTPUT_DIR} not found. Train first!")
